## <span style="color: Violet;"> **Импорт библиотек и Интерфейсы** </style> 

In [1]:
import json
import os
import copy
from abc import ABC, abstractmethod
from typing import List, Optional, Dict, Any

class IUnit(ABC):
    @property
    @abstractmethod
    def name(self) -> str: pass

    @property
    @abstractmethod
    def cost(self) -> int: pass

    @property
    @abstractmethod
    def hp(self) -> int: pass
    
    @abstractmethod
    def take_damage(self, amount: int) -> None: pass

class ILandscape(ABC):
    @property
    @abstractmethod
    def symbol(self) -> str: pass

    @property
    @abstractmethod
    def move_cost(self) -> int: pass

class INeutralObject(ABC):
    @property
    @abstractmethod
    def symbol(self) -> str: pass

    @abstractmethod
    def effect(self, unit: 'IUnit') -> bool:
        """Возвращает True, если объект поглощается/уничтожается после взаимодействия"""
        pass


## <span style="color: Plum;"> **Окружение и Нейтральные объекты** </style> 

In [2]:
# Landscapes

class Plain(ILandscape):
    symbol = '.'
    move_cost = 1

class Forest(ILandscape):
    symbol = '+'
    move_cost = 2

class Mountain(ILandscape):
    symbol = "^"
    move_cost = 99

class Reaver(ILandscape):
    symbol = '~'
    move_cost = 4

# Neutral Objects

class HealingShrine(INeutralObject):
    symbol = 'H'
    def effect(self, unit: IUnit) -> bool:
        hp_before = unit.hp
        unit.hp = min(unit.hp + 20, unit.max_hp)
        hp_restored = unit.hp - hp_before
        print(f"Зел-нага услышали молитвы {unit.name} ")
        print(f"{unit.name} восстановил {hp_restored} HP")

class Rill(INeutralObject):
    symbol = '='
    def effect(self, unit: IUnit) -> bool:
        hp_before = unit.hp
        unit.hp = min(unit.hp + 5, unit.max_hp)
        hp_restored = unit.hp - hp_before
        print(f"{unit.name} выпил водички из родника")
        print(f"{unit.name} восстановил {hp_restored} HP")

class Chest(INeutralObject):
    symbol = 'X'
    def effect(self, unit: IUnit) -> bool:
        print(f"{unit.name} обнаружил сундук, однако удача отвернулась от него: внутри было пусто")
        return True

class Trap(INeutralObject):
    symbol = '_'
    def effect(self, unit: IUnit) -> bool:
        unit.take_damage(100)
        print(f"{unit.name} угодил в ловушку")
        print(f"{unit.name} потерял 100 HP")
        return True

class EnrichedMine(INeutralObject):
    symbol = "M"  
    Mineral_get = 50  
    
    def effect(self, unit: IUnit) -> bool:
        """Добыча золота с рудника"""
        # Проверяем, привязан ли юнит к базе
        if unit.base is not None:
            unit.base.gold += self.Mineral_get
            print(f"[ЛОГ] {unit.name} добыл {self.Mineral_get} минералов с рудника! "
                  f"(Всего у базы: {unit.base.gold})")
        else:
            print(f"[ЛОГ] {unit.name} нашёл рудник, но не может доставить ресурсы (нет базы)")
        
        return False 


## <span style="color: purple;"> **Юниты** </style> 

In [3]:
class Unit(IUnit):
    def __init__(self, name: str, hp: int, attack: int, armor: int, 
                 move_range: int, cost: int = 0):
        self._name = name
        self.max_hp = hp
        self._hp = hp
        self.attack = attack
        self.armor = armor
        self.move_range = move_range
        self._cost = cost  
        self.x = 0
        self.y = 0
        self.base = None

    @property
    def name(self) -> str: 
        return self._name
    
    @property
    def hp(self) -> int: 
        return self._hp
    
    @hp.setter
    def hp(self, value: int): 
        self._hp = max(0, value)

    @property
    def cost(self) -> int:
        return self._cost
    
    @cost.setter
    def cost(self, value: int):
        self._cost = max(0, value)

    def take_damage(self, amount: int) -> None:
        damage = max(0, amount - self.armor)
        self.hp -= damage

    def __add__(self, other: INeutralObject):
        if isinstance(other, INeutralObject):
            is_consumed = other.apply_effect(self)
            return is_consumed
        raise TypeError("Юнит может взаимодействовать только с нейтральными объектами через оператор +")
    
# Единицы ближнего боя
class Zealot(Unit):
    def __init__(self): super().__init__("Зилот", hp=100, attack = 15, armor = 15, move_range = 3, cost=100)
class DarkTemplar(Unit):
    def __init__(self): super().__init__("Тёмный тамплиер", hp=50, attack = 30, armor = 5, move_range = 4, cost=125)
# Единицы дальнего боя
class Dragoon(Unit):
    def __init__(self): super().__init__("Драгун", hp=120, attack = 20, armor = 20, move_range = 3, cost=150)
class Stalker(Unit):
    def __init__(self): super().__init__("Сталкер", hp=100, attack = 15, armor = 10, move_range = 5, cost=125)
# Тяжёлые механические единицы
class Reaver(Unit):
    def __init__(self): super().__init__("Опустошитель", hp=100, attack = 80, armor = 5, move_range = 1, cost=200)
class Immortal(Unit):
    def __init__(self): super().__init__("Бессмертный", hp=150, attack = 20, armor = 40, move_range = 2, cost=200)
class Colossus(Unit):
    def __init__(self): super().__init__("Колосс", hp=200, attack = 40, armor = 20, move_range = 8, cost=600)


## <span style="color: #957B8D;"> **Игровое поле и База** </style> 

In [4]:
import copy
from typing import List, Optional

class Cell:
    """Клетка игрового поля — содержит ландшафт, юнита, нейтральный объект или базу"""
    def __init__(self):
        self.landscape: ILandscape = Plain()
        self.unit: Optional[IUnit] = None
        self.neutral_object: Optional[INeutralObject] = None
        self.base: Optional['GameBase'] = None

class GameBase:
    """База игрока — создаёт юнитов и учитывает их"""
    def __init__(self, x: int, y: int, max_units: int = 5):
        self.x = x
        self.y = y
        self.hp = 500
        self.max_units = max_units
        self.created_units: List[IUnit] = []
        self.gold = 1000 

    def create_unit(self, unit_class) -> Optional[IUnit]:
        """Создание юнита на базе с проверкой золота и лимита"""
        if len(self.created_units) >= self.max_units:
            print(f"Достигнут лимит юнитов на базе! Максимум: {self.max_units}")
            return None
        
        # Создаём временный экземпляр для проверки стоимости
        temp_unit = unit_class()
        
        # Проверка стоимости
        if temp_unit.cost > self.gold:
            print(f"Недостаточно минералов! Нужно {temp_unit.cost}, у вас {self.gold}")
            return None
        
        # Списываем золото
        self.gold -= temp_unit.cost
        
        # Создаём настоящего юнита
        unit = unit_class()
        unit.base = self  # ← Привязываем юнита к базе
        unit.x, unit.y = self.x, self.y
        self.created_units.append(unit)
        
        print(f"✓ Создан юнит: {unit.name} (стоимость: {unit.cost} минералов)")
        return unit

    def remove_unit(self, unit: IUnit):
        """Удаление юнита из списка созданных"""
        if unit in self.created_units:
            self.created_units.remove(unit)
            print(f"Юнит {unit.name} удалён из списка базы")

    def get_status(self) -> str:
        """Получение статуса базы в виде строки"""
        return (f"База на ({self.x}, {self.y}) | HP: {self.hp} | "
                f"Юнитов: {len(self.created_units)}/{self.max_units} | "
                f"Минералы: {self.gold}")

class GameField:
    """Игровое поле — контейнер для всех объектов"""
    
    def __init__(self, width: int, height: int, max_objects: int = 100):
        self.width = width
        self.height = height
        self.max_objects = max_objects
        self.grid: List[List[Cell]] = [[Cell() for _ in range(width)] for _ in range(height)]
        self.bases: List[GameBase] = []
        self._object_count = 0

    def set_landscape(self, x: int, y: int, landscape: ILandscape):
        """Установка типа ландшафта на клетке"""
        if 0 <= x < self.width and 0 <= y < self.height:
            self.grid[y][x].landscape = landscape

    def add_unit(self, unit: IUnit, x: int, y: int) -> bool:
        """Добавление юнита на поле с контролем лимита объектов"""
        if self._object_count >= self.max_objects:
            print(f"Достигнут лимит объектов на поле! Максимум: {self.max_objects}")
            return False
        
        if not (0 <= x < self.width and 0 <= y < self.height):
            print("Координаты вне поля!")
            return False
            
        if self.grid[y][x].unit is not None:
            print("Клетка уже занята другим юнитом!")
            return False
            
        self.grid[y][x].unit = unit
        unit.x, unit.y = x, y
        self._object_count += 1
        return True

    def remove_unit(self, unit: IUnit):
        """Удаление юнита с поля"""
        if 0 <= unit.y < self.height and 0 <= unit.x < self.width:
            if self.grid[unit.y][unit.x].unit == unit:
                self.grid[unit.y][unit.x].unit = None
                self._object_count -= 1

    def add_base(self, base: GameBase):
        """Размещение базы на поле"""
        self.bases.append(base)
        self.grid[base.y][base.x].base = base
        self._object_count += 1

    def move_unit(self, unit: IUnit, new_x: int, new_y: int) -> bool:
        """Перемещение юнита с учётом ландшафта и лимитов"""

        if unit.hp <= 0:
            print(f" {unit.name} мёртв и не может перемещаться!")
            return False
          
        if not (0 <= new_x < self.width and 0 <= new_y < self.height):
            print("Целевая клетка находится за пределами поля!")
            return False
        
        target_cell = self.grid[new_y][new_x]
        
        if target_cell.unit is not None:
            print("Клетка занята другим юнитом!")
            return False
            
        if target_cell.base is not None:
            print("Нельзя встать на базу!")
            return False

        # Проверка дальности хода
        distance = abs(new_x - unit.x) + abs(new_y - unit.y)
        if distance > unit.move_range:
            print(f"Слишком далеко! Дальность хода юнита: {unit.move_range}, расстояние: {distance}")
            return False
        
        # Проверка проходимости ландшафта
        if target_cell.landscape.move_cost > unit.move_range:
            print(f"Местность непроходима! Стоимость: {target_cell.landscape.move_cost}")
            return False

        # Перемещение юнита
        self.grid[unit.y][unit.x].unit = None
        unit.x, unit.y = new_x, new_y
        target_cell.unit = unit
        
        # Взаимодействие с нейтральным объектом через перегрузку оператора +
        if target_cell.neutral_object is not None:
            is_consumed = unit + target_cell.neutral_object
            if is_consumed:
                target_cell.neutral_object = None
                self._object_count -= 1
                print(f"Нейтральный объект на клетке ({new_x}, {new_y}) использован")
                
        return True
    def cleanup_dead_units(self) -> List[str]:
        """Удаление мёртвых юнитов с поля и из баз"""
        dead_units = []
            
        for base in self.bases:
            # Создаём копию списка, чтобы безопасно удалять элементы
            for unit in base.created_units[:]:
                if unit.hp <= 0:
                     # Удаляем с поля
                    if 0 <= unit.y < self.height and 0 <= unit.x < self.width:
                        if self.grid[unit.y][unit.x].unit == unit:
                            self.grid[unit.y][unit.x].unit = None
                            self._object_count -= 1
                        
                    # Удаляем из базы
                    base.remove_unit(unit)
                    dead_units.append(unit.name)
                    print(f"{unit.name} погиб и удалён из игры")
            
        return dead_units

    def add_neutral_object(self, x: int, y: int, obj: INeutralObject):
        """Добавление нейтрального объекта на поле"""
        if not (0 <= x < self.width and 0 <= y < self.height):
            print("Координаты вне поля!")
            return False
            
        if self.grid[y][x].neutral_object is not None:
            print("На этой клетке уже есть нейтральный объект!")
            return False
            
        self.grid[y][x].neutral_object = obj
        self._object_count += 1
        return True
    
    def copy_field(self) -> 'GameField':
        """Глубокое копирование поля со всеми объектами"""
        return copy.deepcopy(self)

    def render(self) -> str:
        """Отрисовка поля в виде строки"""
        # Определяем ширину ячейки
        cell_width = 3 if self.width > 10 or self.height > 10 else 2
        
        # Заголовок с координатами X
        header = "   "
        for i in range(self.width):
            if cell_width == 3:
                header += f"{i:2d} "  # Двузначные числа с выравниванием
            else:
                header += f"{i} "
        result = header + "\n"
        
        # Верхняя граница
        result += "  +" + ("--" * self.width if cell_width == 2 else "---" * self.width) + "+\n"
        
        # Отрисовка строк
        for y in range(self.height):
            # Координата Y
            if cell_width == 3:
                row_str = f"{y:2d}|"
            else:
                row_str = f"{y % 10}|"
            
            # Клетки
            for x in range(self.width):
                cell = self.grid[y][x]
                
                # Определяем символ
                if cell.base:
                    symbol = "B"
                elif cell.unit:
                    name = cell.unit.name
                    if "Зилот" in name:
                        symbol = "Z"
                    elif "Тёмный Тамплиер" in name:
                        symbol = "T"
                    elif "Драгун" in name:
                        symbol = "D"
                    elif "Сталкер" in name:
                        symbol = "S"
                    elif "Опустошитель" in name:
                        symbol = "R"
                    elif "Бессмертный" in name:
                        symbol = "I"
                    elif "Колосс" in name:
                        symbol = "C"
                    else:
                        symbol = "U"
                elif cell.neutral_object:
                    symbol = cell.neutral_object.symbol
                else:
                    symbol = cell.landscape.symbol
                
                # Добавляем символ с правильным форматированием
                if cell_width == 3:
                    row_str += f" {symbol} "
                else:
                    row_str += f"{symbol} "
            
            result += row_str + "|\n"
        
        # Нижняя граница
        result += "  +" + ("--" * self.width if cell_width == 2 else "---" * self.width) + "+"
        
        return result

## **Генерация**

In [5]:
import random
from typing import List, Tuple, Optional

class MapGenerator:
    """Генератор случайных карт для игры"""
    
    # Соотношение ландшафтов (должно давать в сумме 100)
    Landscape_weights = {
        'plain': 60,     # 60% равнин
        'forest': 25,    # 25% лесов
        'mountain': 15   # 15% гор
    }
    
    # Количество нейтральных объектов каждого типа
    Neutral_count = {
        HealingShrine: 1,
        Trap: 6,
        Chest: 2,
        Rill: 5,
        EnrichedMine: 3
    }
    
    def __init__(self, seed: Optional[int] = None):
        """
        Инициализация генератора
        :param seed: сид для воспроизводимости (None = случайный)
        """
        self.seed = seed
        if seed is not None:
            random.seed(seed)
        else:
            self.seed = random.randint(0, 999999)
            random.seed(self.seed)
        print(f"Используется сид карты: {self.seed}")
    
    def generate_landscape(self, field: 'GameField') -> None:
        """Случайная генерация ландшафта"""
        # Создаём список ландшафтов согласно весам
        landscape_pool = []
        for landscape_type, weight in self.Landscape_weights.items():
            if landscape_type == 'plain':
                landscape_pool.extend([Plain()] * weight)
            elif landscape_type == 'forest':
                landscape_pool.extend([Forest()] * weight)
            elif landscape_type == 'mountain':
                landscape_pool.extend([Mountain()] * weight)
        
        # Заполняем карту случайным ландшафтом
        for y in range(field.height):
            for x in range(field.width):
                landscape = random.choice(landscape_pool)
                field.set_landscape(x, y, landscape)
    
    def _get_empty_cells(self, field: 'GameField', 
                         exclude_zones: List[Tuple[int, int]] = None) -> List[Tuple[int, int]]:
        """Получить список пустых клеток (без юнитов, баз и нейтральных объектов)"""
        exclude_zones = exclude_zones or []
        empty_cells = []
        
        for y in range(field.height):
            for x in range(field.width):
                cell = field.grid[y][x]
                if (cell.unit is None and 
                    cell.base is None and 
                    cell.neutral_object is None and
                    (x, y) not in exclude_zones):
                    empty_cells.append((x, y))
        
        return empty_cells
    
    def generate_neutral_objects(self, field: 'GameField', 
                                  exclude_zones: List[Tuple[int, int]] = None) -> None:
        """Случайное размещение нейтральных объектов"""
        exclude_zones = exclude_zones or []
        
        for obj_class, count in self.Neutral_count.items():
            for _ in range(count):
                empty_cells = self._get_empty_cells(field, exclude_zones)
                if not empty_cells:
                    print(f"Не удалось разместить {obj_class.__name__} — нет свободных клеток")
                    break
                
                x, y = random.choice(empty_cells)
                field.add_neutral_object(x, y, obj_class())
                exclude_zones.append((x, y))  # Чтобы не размещать два объекта в одной клетке
    
    def generate_map(self, width: int = 12, height: int = 12, 
                     base_position: Tuple[int, int] = (1, 1),
                     safe_zone_radius: int = 2) -> 'GameField':
        """
        Полная генерация карты
        :param width: ширина карты
        :param height: высота карты
        :param base_position: координаты базы (x, y)
        :param safe_zone_radius: радиус безопасной зоны вокруг базы (без гор)
        :return: сгенерированное игровое поле
        """
        field = GameField(width=width, height=height, max_objects=100)
        
        # 1. Генерируем ландшафт
        self.generate_landscape(field)
        
        # 2. Определяем безопасную зону вокруг базы (там не должно быть гор)
        safe_zone = []
        bx, by = base_position
        for dy in range(-safe_zone_radius, safe_zone_radius + 1):
            for dx in range(-safe_zone_radius, safe_zone_radius + 1):
                nx, ny = bx + dx, by + dy
                if 0 <= nx < width and 0 <= ny < height:
                    safe_zone.append((nx, ny))
                    # В безопасной зоне всегда равнина
                    field.set_landscape(nx, ny, Plain())
        
        # 3. Размещаем базу
        base = GameBase(x=bx, y=by, max_units=7)
        field.add_base(base)
        
        # 4. Размещаем нейтральные объекты (избегая безопасной зоны)
        self.generate_neutral_objects(field, exclude_zones=safe_zone)
        
        print(f"Карта {width}x{height} сгенерирована!")
        print(f"База размещена на ({bx}, {by})")
        
        return field

## <span style="color: #702963;"> **Игровой движок и Интерфейс** </style>

In [6]:
class GameEngine:
    """Игровой движок — консольный интерфейс для управления игрой"""
    
    def __init__(self, seed: Optional[int] = None):
        # Инициализация логгера
        self.logger = GameLogger("game_session.json")
        self.logger.log("ЗАПУСК_ИГРЫ", "Приложение инициализировано в Jupyter")
        
        # Настройка карты с возможностью seed
        self._setup_map(seed=seed)
        self.is_running = True

    def _setup_map(self, seed: Optional[int] = None):
        """Инициализация карты через генератор"""
        # Создаём генератор карт
        generator = MapGenerator(seed=seed)
        
        # Генерируем карту 20x20 с базой в (1, 1)
        self.field = generator.generate_map(
            width=20, 
            height=20, 
            base_position=(1, 1),
            safe_zone_radius=2
        )
        
        # Сохраняем ссылку на базу
        self.player_base = self.field.bases[0]
        
        # Создаём начальных юнитов
        self._spawn_unit(Zealot)
        self._spawn_unit(Dragoon)

    def _spawn_unit(self, unit_class):
        """Создание юнита на базе и размещение на поле"""
        unit = self.player_base.create_unit(unit_class)
        if unit:
            placed = False
            for dy in range(-1, 2):
                for dx in range(-1, 2):
                    new_x, new_y = self.player_base.x + dx, self.player_base.y + dy
                    if self.field.add_unit(unit, new_x, new_y):
                        placed = True
                        self.logger.log("СОЗДАНИЕ_ЮНИТА", f"{unit.name} создан на базе")
                        break
                if placed:
                    break
            
            if not placed:
                print("Не удалось разместить юнита — все клетки вокруг базы заняты!")
                self.player_base.remove_unit(unit)

    def handle_command(self, cmd: str):
        """Обработка консольной команды"""
        parts = cmd.lower().split()
        if not parts:
            return

        action = parts[0]
        
        # Команда spawn: создание юнита
        if action == "spawn" and len(parts) > 1:
            unit_type = parts[1]
            unit_map = {
                "zealot": Zealot,
                "dark": DarkTemplar,
                "dragoon": Dragoon,
                "stalker": Stalker,
                "reaver": Reaver,
                "immortal": Immortal,
                "colossus": Colossus
            }
            
            if unit_type in unit_map:
                self._spawn_unit(unit_map[unit_type])
            else:
                print("Неизвестный тип юнита!")
                print("Доступны: zealot, dark, dragoon, stalker, reaver, immortal, colossus")
            
        # Команда move: перемещение юнита 
        elif action == "move" and len(parts) >= 3:
            try:
                # Если указан индекс юнита: move <индекс> <x> <y>
                if len(parts) == 4:
                    unit_idx = int(parts[1])
                    new_x, new_y = int(parts[2]), int(parts[3])
                else:
                    unit_idx = 0
                    new_x, new_y = int(parts[1]), int(parts[2])
                
                if not self.player_base.created_units:
                    print("Нет активных юнитов для перемещения!")
                    return
                
                if unit_idx >= len(self.player_base.created_units):
                    print(f"Неверный индекс юнита! Доступно: 0-{len(self.player_base.created_units)-1}")
                    return
                
                unit = self.player_base.created_units[unit_idx]
                
                if self.field.move_unit(unit, new_x, new_y):
                    self.field.cleanup_dead_units()
                    self.logger.log("ПЕРЕМЕЩЕНИЕ_ЮНИТА", f"{unit.name} перемещён на ({new_x}, {new_y})")
                    print(f"{unit.name} перемещён на ({new_x}, {new_y})")
                else:
                    print(f"Невозможно переместить {unit.name}")
                    
            except ValueError:
                print("Неверный формат! Используйте: move [индекс] <x> <y>")
                
        # Команда status: состояние базы и юнитов
        elif action == "status":
            dead = self.field.cleanup_dead_units()
            if dead:
                print(f"\nПогибшие юниты: {', '.join(dead)}\n")
            
            print("\n" + "="*50)
            print("Состояние базы")
            print("="*50)
            print(self.player_base.get_status())
            
            if self.player_base.created_units:
                print("\nЮНИТЫ:")
                for idx, unit in enumerate(self.player_base.created_units):
                    status = "☠️ МЁРТВ" if unit.hp <= 0 else "жив"
                    print(f"  [{idx}] {unit.name} на ({unit.x}, {unit.y}) | "
                        f"HP: {unit.hp}/{unit.max_hp} | "
                        f"Атака: {unit.attack} | "
                        f"Броня: {unit.armor} | "
                        f"Ход: {unit.move_range} | "
                        f"Цена: {unit.cost} | {status}")
            else:
                print("\nНет активных юнитов")
            print("="*50 + "\n")

        # Команда save: сохранение игры
        elif action == "save":
            filename = "save.json"
            if len(parts) > 1:
                filename = parts[1]  # Можно указать своё имя файла
            self.save_game(filename)
            
        # Команда load: загрузка игры
        elif action == "load":
            filename = "save.json"
            if len(parts) > 1:
                filename = parts[1]  # Можно указать своё имя файла
            self.load_game(filename)   
             
        # Команда gather: сбор ресурсов
        elif action == "gather":
            if not self.player_base.created_units:
                print("Нет юнитов для сбора ресурсов!")
                return
            
            # По умолчанию собираем первым юнитом
            unit_idx = 0
            if len(parts) > 1:
                try:
                    unit_idx = int(parts[1])
                except ValueError:
                    print("Неверный индекс юнита!")
                    return
            
            if unit_idx >= len(self.player_base.created_units):
                print(f"Неверный индекс! Доступно: 0-{len(self.player_base.created_units)-1}")
                return
            
            unit = self.player_base.created_units[unit_idx]
            
            # Проверяем, есть ли на клетке юнита рудник
            cell = self.field.grid[unit.y][unit.x]
            if cell.neutral_object is not None and isinstance(cell.neutral_object, EnrichedMine):
                # Взаимодействуем с рудником через перегрузку +
                unit + cell.neutral_object
                self.logger.log("СБОР_РЕСУРСОВ", f"{unit.name} добыл минералы с рудника")
            else:
                print(f"На клетке ({unit.x}, {unit.y}) нет рудника для сбора!")
                print("Переместите юнита на клетку с рудником (символ 'E')")
        
        # Команда copy: копирование поля
        elif action == "copy":
            new_field = self.field.copy_field()
            self.logger.log("КОПИРОВАНИЕ_ПОЛЯ", "Создана глубокая копия игрового поля")
            print(f"Поле успешно скопировано. Размеры: {new_field.width}x{new_field.height}")
            
        # Команда map: показать карту
        elif action == "map":
            print("\n" + self.field.render() + "\n")
            
        # Команда help: справка
        elif action == "help":
            print("\n" + "="*50)
            print("Справка по командам")
            print("="*50)
            print("spawn <тип>       — создать юнита")
            print("  Типы: zealot, dark, dragoon, stalker, reaver, immortal, colossus")
            print("move <x> <y>      — переместить первого юнита")
            print("move <индекс> <x> <y> — переместить юнита по индексу")
            print("gather [индекс]   — собрать ресурсы с рудника")
            print("status            — показать состояние базы и юнитов")
            print("map               — показать карту")
            print("copy              — скопировать поле")
            print("save [файл]       — сохранить игру (по умолчанию: save.json)")
            print("load [файл]       — загрузить игру (по умолчанию: save.json)")
            print("help              — показать эту справку")
            print("quit              — выйти из игры")
            print("="*50 + "\n")
            
        # Команда quit: выход
        elif action == "quit":
            self.is_running = False
            self.logger.log("ЗАВЕРШЕНИЕ_ИГРЫ", "Приложение завершено пользователем")
            print("Спасибо за игру! Логи сохранены в game_session.json")
            
        else:
            print("Неизвестная команда! Введите 'help' для справки.")
    
    def run(self):
        """Запуск игрового цикла"""
        print("\n" + "="*50)
        print("Starctaft 0.5 - Протоссы")
        print("="*50)
        print("Введите 'help' для списка команд, 'quit' для выхода.")
        print("="*50 + "\n")
        
        while self.is_running:
            print(self.field.render())
            cmd = input("\nВведите команду > ").strip()
            self.handle_command(cmd)
    def save_game(self, filename: str = "save.json"):
        """Сохранение состояния игры в JSON-файл"""
        manager = JsonManager(filename)
        
        # Сохраняем данные базы
        base_data = {
            "gold": self.player_base.gold,
            "hp": self.player_base.hp,
            "max_units": self.player_base.max_units,
            "position": {"x": self.player_base.x, "y": self.player_base.y}
        }
        manager.add("base", base_data)
        
        # Сохраняем юнитов
        units_data = []
        for unit in self.player_base.created_units:
            unit_data = {
                "class_name": type(unit).__name__,  # Имя класса (Zealot, Dragoon и т.д.)
                "hp": unit.hp,
                "max_hp": unit.max_hp,
                "attack": unit.attack,
                "armor": unit.armor,
                "move_range": unit.move_range,
                "cost": unit.cost,
                "position": {"x": unit.x, "y": unit.y}
            }
            units_data.append(unit_data)
        
        manager.add("units", units_data)
        
        # Сохраняем сид карты (если нужен для воспроизведения)
        manager.add("seed", getattr(self, '_seed', None))
        
        self.logger.log("СОХРАНЕНИЕ_ИГРЫ", f"Игра сохранена в {filename}")
        print(f"Игра сохранена в {filename}")
        print(f"   База: {base_data['gold']} минералов, HP: {base_data['hp']}")
        print(f"   Юнитов: {len(units_data)}")

    def load_game(self, filename: str = "save.json"):
        """Загрузка состояния игры из JSON-файла"""
        manager = JsonManager(filename)
        
        # Проверяем существование файла
        if not manager.read():
            print(f"Файл сохранения {filename} не найден или пуст!")
            return False
        
        # Загружаем данные базы
        base_data = manager.get("base")
        if not base_data:
            print("Данные базы не найдены в файле сохранения!")
            return False
        
        # Восстанавливаем базу
        self.player_base.gold = base_data["gold"]
        self.player_base.hp = base_data["hp"]
        self.player_base.max_units = base_data["max_units"]
        
        # Удаляем старых юнитов с поля
        for unit in self.player_base.created_units[:]:
            if 0 <= unit.y < self.field.height and 0 <= unit.x < self.field.width:
                if self.field.grid[unit.y][unit.x].unit == unit:
                    self.field.grid[unit.y][unit.x].unit = None
                    self.field._object_count -= 1
        self.player_base.created_units.clear()
        
        # Загружаем юнитов
        units_data = manager.get("units", [])
        
        # Маппинг имён классов на сами классы
        unit_classes = {
            "Zealot": Zealot,
            "DarkTemplar": DarkTemplar,
            "Dragoon": Dragoon,
            "Stalker": Stalker,
            "Reaver": Reaver,
            "Immortal": Immortal,
            "Colossus": Colossus
        }
        
        for unit_data in units_data:
            class_name = unit_data["class_name"]
            
            # Проверяем, известен ли класс
            if class_name not in unit_classes:
                print(f"Неизвестный тип юнита: {class_name}, пропускаем")
                continue
            
            # Создаём юнита через класс (он сам установит базовые характеристики)
            unit = unit_classes[class_name]()
            
            # Переопределяем характеристики из сохранения
            unit.hp = unit_data["hp"]
            unit.max_hp = unit_data["max_hp"]
            unit.attack = unit_data["attack"]
            unit.armor = unit_data["armor"]
            unit.move_range = unit_data["move_range"]
            unit.cost = unit_data["cost"]
            
            # Размещаем на поле
            pos = unit_data["position"]
            x, y = pos["x"], pos["y"]
            
            if 0 <= x < self.field.width and 0 <= y < self.field.height:
                if self.field.grid[y][x].unit is None:
                    self.field.grid[y][x].unit = unit
                    unit.x, unit.y = x, y
                    self.field._object_count += 1
                    self.player_base.created_units.append(unit)
                    unit.base = self.player_base
                else:
                    print(f"Клетка ({x}, {y}) занята, юнит {unit.name} не размещён")
            else:
                print(f"Координаты ({x}, {y}) вне поля, юнит {unit.name} не размещён")
        
        self.logger.log("ЗАГРУЗКА_ИГРЫ", f"Игра загружена из {filename}")
        print(f"Игра загружена из {filename}")
        print(f"   База: {self.player_base.gold} минералов, HP: {self.player_base.hp}")
        print(f"   Юнитов: {len(self.player_base.created_units)}")
        
        return True


## <span style="color: #CDA4DE;"> **Логирование**</style>

In [7]:
from datetime import datetime 

class GameLogger:
    def __init__(self, log_filepath: str = "game_log.json"):
        self.manager = JsonManager(log_filepath)
        if self.manager.get("logs") is None:
            self.manager.add("logs", [])

    def log(self, action: str, details: str):
        logs = self.manager.get("logs")
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        log_entry = {"время": timestamp, "действие": action, "описание": details}
        logs.append(log_entry)
        self.manager.update("logs", logs)

## <span style="color: Lavender;"> **Задача 19(Работа с JSON)** </style> 

In [8]:
import os
import json
from typing import Any
class JsonManager:
    "Инициализация файла"
    def __init__(self, filepath: str):
        self.filepath = filepath
        if not os.path.exists(self.filepath):
            self.write({})

    "Функция для чтения файла"
    def read(self) -> dict:
        try:
            with open(self.filepath, 'r', encoding='utf-8') as f:
                return json.load(f)
        except (FileNotFoundError, json.JSONDecodeError):
            return {}
            
    "Функция для полной перезаписи файла"
    def write(self, data: dict) -> None:
        if not isinstance(data, dict):
            raise TypeError("Корень JSON должен быть словарём (dict) для корректной работы методов.")
        with open(self.filepath, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)

    "Функция для добавление новой пары ключ-значение"
    def add(self, key: str, value: Any) -> None:
        data = self.read()
        if key in data:
            raise ValueError(f"Ключ {key} уже существует, используйте update().")
        data[key]=value
        self.write(data)

    "Функция для обновления ключа" 
    def update(self, key: str, value: Any) -> None:
        data = self.read()
        if key not in data:
            raise KeyError(f"Ключ {key} не найден")
        data[key] = value
        self.write(data)

    "Функция для удаления ключа"
    def remove(self, key: str, value: Any) -> None:
            data = self.read()
            if key not in data:
                raise KeyError(f"Ключ {key} не найден")
            del data[key]
            self.write(data)
    "Функция для получения ключа"
    def get(self, key: str, default=None):
        return self.read().get(key, default)

## <span style="color: #4B0082;"> **Запуск** </style> 

In [9]:
print("Инициализация игрового мира...")
game = GameEngine()
print("Игра готова к запуску!\n")
game.run()

Инициализация игрового мира...
Используется сид карты: 576355
Карта 20x20 сгенерирована!
База размещена на (1, 1)
✓ Создан юнит: Зилот (стоимость: 100 минералов)
✓ Создан юнит: Драгун (стоимость: 150 минералов)
Клетка уже занята другим юнитом!
Игра готова к запуску!


Starctaft 0.5 - Протоссы
Введите 'help' для списка команд, 'quit' для выхода.

    0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 
  +------------------------------------------------------------+
 0| Z  D  .  .  .  .  .  ^  ^  .  .  +  +  ^  .  .  .  .  .  . |
 1| .  B  .  .  ^  _  .  ^  +  +  .  .  .  .  .  ^  .  .  +  + |
 2| .  .  .  .  .  .  +  .  .  +  .  +  .  _  .  .  .  +  M  + |
 3| .  .  .  .  +  .  ^  .  ^  .  +  .  +  +  .  .  .  +  .  ^ |
 4| ^  ^  ^  .  .  .  +  .  +  .  .  +  .  ^  =  +  .  .  .  . |
 5| .  +  +  .  +  .  ^  +  ^  .  ^  +  .  .  .  .  .  .  .  + |
 6| ^  .  .  ^  ^  =  .  +  .  .  +  .  +  .  +  .  .  ^  .  . |
 7| .  .  ^  .  .  +  ^  +  .  .  +  M  +  +  +  +  +  .  ^  . |
 8|

Спасибо за игру! Логи сохранены в game_session.json


## <span style="color: #4B0082;"> **Виджеты** </style> 

In [10]:
from ipywidgets import interact, widgets
from IPython.display import display, clear_output

# Создаём текстовое поле для ввода команд
command_input = widgets.Text(
    value='',
    placeholder='Введите команду (например: spawn zealot)',
    description='Команда:',
    layout=widgets.Layout(width='50%')
)

# Создаём кнопку выполнения
run_button = widgets.Button(
    description="Выполнить",
    button_style='success',
    icon='play'
)

# Создаём область вывода
output = widgets.Output(
    layout=widgets.Layout(border='1px solid gray', padding='10px')
)

def on_button_clicked(b):
    with output:
        clear_output()
        cmd = command_input.value.strip()
        if not cmd:
            print("Введите команду!")
            return
        
        print(f">>> {cmd}\n")
        game.handle_command(cmd)
        print("\n" + game.field.render())
        
        # Очищаем поле ввода после выполнения
        command_input.value = ''

run_button.on_click(on_button_clicked)

def on_text_submit(change):
    on_button_clicked(None)

command_input.on_submit(on_text_submit)

print("Старкрафт 0.5 — Протоссы")
print("Введите команду и нажмите 'Выполнить' или Enter\n")
display(command_input, run_button, output)

with output:
    print("Начальное состояние карты:")
    print(game.field.render())

Старкрафт 0.5 — Протоссы
Введите команду и нажмите 'Выполнить' или Enter



C:\Users\user\AppData\Local\Temp\ipykernel_21972\2140096043.py:44: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  command_input.on_submit(on_text_submit)


Text(value='', description='Команда:', layout=Layout(width='50%'), placeholder='Введите команду (например: spa…

Button(button_style='success', description='Выполнить', icon='play', style=ButtonStyle())

Output(layout=Layout(border_bottom='1px solid gray', border_left='1px solid gray', border_right='1px solid gra…